In [1]:
# Get raw data
with open('input/16.txt', 'r') as f:
    rawinput = f.read().strip()

In [2]:
from math import prod

In [ ]:
# Part 1
class BitFeed(object):
    def __init__(self, hexa):
        self.binary = ''.join([('000'+bin(int(i,16))[2:])[-4:] for i in hexa])
        
    def from_bits(self, n, as_bits=False):
        bits = self.binary[self.ptr:self.ptr+n]
        self.ptr += n
        return bits if as_bits else int(bits, 2)
    
    def parse_literal(self):
        lit_bits = ''
        while True:
            lit_bits += (z:=self.from_bits(5, True))[1:]
            if z[0]=='0':
                break
        return int(lit_bits,2)
    
    def parse(self):
        self.ptr = 0
        return self.parse_rec(max_ptr=len(self.binary))
    
    def parse_rec(self, n_packets=None, max_ptr=None):
        result = []
        pkct = 0
        while ((n_packets is None or pkct < n_packets) 
               and (max_ptr is None or self.ptr < max_ptr)
               and {*self.binary[self.ptr:]} != {'0'}):
            version = self.from_bits(3)
            type_id = self.from_bits(3)
            if type_id==4:
                lit = self.parse_literal()
                result += [[version, type_id, lit]]
            else:
                len_type = self.from_bits(1)
                if len_type==0:
                    sp_end = self.from_bits(15) + self.ptr
                    result += [[version, type_id, self.parse_rec(max_ptr=sp_end)]]
                else:
                    sp_num = self.from_bits(11)
                    result += [[version, type_id, self.parse_rec(n_packets=sp_num)]]
            pkct += 1
            
        return result
    
sum_versions = lambda x: sum([i+(0 if j==4 else sum_versions(k)) 
                              for i,j,k in x])

signal = BitFeed(rawinput)
sum_versions(signal.parse())

960

In [ ]:
# Part 2
func = {
    0: lambda x: sum(x),
    1: lambda x: prod(x),
    2: lambda x: min(x),
    3: lambda x: max(x),
    4: lambda x: x,
    5: lambda x: int(x[0]>x[1]),
    6: lambda x: int(x[0]<x[1]),
    7: lambda x: int(x[0]==x[1]),
}
evaluate = lambda x: [func[j](k if j==4 else evaluate(k)) for i,j,k in x]
    
evaluate(signal.parse())[0]

12301926782560